# 06 Execution Simulator

Purpose: compare market, limit, TWAP and VWAP execution cost assumptions.

Data source: `data/processed/execution_comparison.parquet` generated from liquidity diagnostics.


## Methodology Summary

The simulator combines spread crossing, commission bps, slippage, a market impact proxy and fill-rate assumptions. It does not model broker routing, queue position or real order placement.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')

from russian_markets_lab.analytics.execution import explain_execution_assumptions


In [ ]:
execution, metadata = load_dataset('execution_comparison')
if execution.empty:
    show_missing('execution_comparison')
else:
    display(execution)


## Cost Comparison


In [ ]:
if not execution.empty and {'execution_style', 'total_cost_bps'}.issubset(execution.columns):
    fig = px.bar(execution, x='execution_style', y='total_cost_bps', color='execution_risk' if 'execution_risk' in execution.columns else None, title='Execution cost comparison')
    fig.show()
    display(explain_execution_assumptions())


## Key Observations

Record which assumptions drive costs, which styles have lower fill rate and how capacity constraints enter the model.


## Limitations And Disclaimer

Execution estimates are simplified research diagnostics. There is no broker integration or real order sending.
